In [1]:
from CosmiXs import annihilation_spectrum
from scipy.integrate import simpson

spec = annihilation_spectrum(50, 'b', 'Gamma', path='CosmiXs')

mask = (spec[:, 0] > 0.5) & (spec[:, 0] < 5) # 0.5 to 5 GeV
simpson(y=spec[:, 1][mask], x=spec[:, 0][mask])

np.float64(12.790063285591886)

### Question 1.2.1 (Section 1.2)
Using CosmiXs, compute $N_\gamma^{DM}=\int_{E_{\min}}^{E_{\max}} dE\, dN_\gamma/dE$ for $m_{DM}=100$ GeV
in the Fermi GCE range $E_{\min}=0.5$ GeV to $E_{\max}=5$ GeV, for channels:
- $e^+e^-$ (`e`)
- $\mu^+\mu^-$ (`mu`)
- $b\bar{b}$ (`b`)

In [2]:
# Solution 1.2.1
from CosmiXs import annihilation_spectrum
from scipy.integrate import simpson
import numpy as np

mDM = 100.0  # GeV
Emin, Emax = 0.5, 5.0  # GeV (Fermi GCE range)
channels = ['e', 'mu', 'b']

print(f'mDM = {mDM} GeV, integration range = [{Emin}, {Emax}] GeV')
for ch in channels:
    spec = annihilation_spectrum(mDM, ch, 'Gamma', path='CosmiXs')
    E = spec[:, 0]
    dNdE = spec[:, 1]

    mask = (E >= Emin) & (E <= Emax)
    Ngamma = simpson(y=dNdE[mask], x=E[mask])

    print(f'channel {ch:>2}: N_gamma(0.5-5 GeV) = {Ngamma:.6f}')


mDM = 100.0 GeV, integration range = [0.5, 5.0] GeV
channel  e: N_gamma(0.5-5 GeV) = 0.263913
channel mu: N_gamma(0.5-5 GeV) = 0.155009
channel  b: N_gamma(0.5-5 GeV) = 17.414150


### Question 1.3.1 (Section 1.3)
Using $\Phi_\gamma^{\rm obs}=1.6\times10^{-6}$ photons/cm$^2$/s (0.5-5 GeV, $20^\circ$ ROI),
test DM parameterizations consistent with relic-density inspired thermal value $\langle\sigma v\rangle\approx 3\times10^{-26}$ cm$^3$/s.
Check leptonic channels viability and cored vs cusped profiles.

In [3]:
# Solution 1.3.1
import numpy as np
from scipy.integrate import quad, simpson
from scipy.optimize import bisect
from CosmiXs import annihilation_spectrum

# Constants
kpcincm = 3.0857e21
MSuninGeV = 1.115e57
rSun = 8.277
rhoSun = 0.4
r200 = 220.2

# Observed GCE flux in 0.5-5 GeV, within 20 deg
phi_obs = 1.6e-6  # photons / cm^2 / s
Emin, Emax = 0.5, 5.0

def rhogNFW(r, rhos, rs, gamma):
    x = r / rs
    return rhos / (x**gamma * (1 + x)**(3 - gamma))

def rhoiso(r, rhos, rs):
    return rhos/(1+(r/rs)**2)

def rhos(rho_func, rs, *args):
    return rhoSun / rho_func(rSun, 1.0, rs, *args)

def DeltaM200(rho_func, rs, *args):
    rhos_val = rhos(rho_func, rs, *args)
    def integrand(r):
        return r**2 * rho_func(r, rhos_val, rs, *args)
    M200 = 4 * np.pi * quad(integrand, 0, r200, epsrel=1e-5)[0]
    M200 *= kpcincm**3 / MSuninGeV
    return M200 - 1e12

def rs(rho_func, *args):
    return bisect(lambda rr, *aa: DeltaM200(rho_func, rr, *aa), 0.1, 50, args=(args), xtol=1e-6)

def J_los(theta, rho_func, rhos_val, rs_val, *args, D=rSun):
    def integrand(s):
        r = np.sqrt(D**2 + s**2 - 2*s*D*np.cos(theta))
        return rho_func(r, rhos_val, rs_val, *args)**2
    return kpcincm * quad(integrand, 0, 400, epsrel=1e-4)[0]

def Jfactor(thetamax, rho_func, rhos_val, rs_val, *args, D=rSun):
    def integrand(theta):
        return J_los(theta, rho_func, rhos_val, rs_val, *args, D=D) * 2*np.pi*np.sin(theta)
    return quad(integrand, 0, thetamax, epsrel=1e-4)[0]

def Ngamma(m, ch):
    spec = annihilation_spectrum(m, ch, 'Gamma', path='CosmiXs')
    E = spec[:,0]
    dNdE = spec[:,1]
    mask = (E >= Emin) & (E <= Emax)
    return simpson(y=dNdE[mask], x=E[mask])

def phi_pred(sigmav, m, Ng, J):
    return (1/(4*np.pi)) * (sigmav/(2*m*m)) * Ng * J

def sigmav_required(phi, m, Ng, J):
    return phi * 8*np.pi*m*m/(Ng*J)

# J-factors in 20 deg ROI
theta20 = np.radians(20)
rs_nfw = rs(rhogNFW, 1.0)
rhos_nfw = rhos(rhogNFW, rs_nfw, 1.0)
J_nfw = Jfactor(theta20, rhogNFW, rhos_nfw, rs_nfw, 1.0)

rs_cusp = rs(rhogNFW, 1.26)
rhos_cusp = rhos(rhogNFW, rs_cusp, 1.26)
J_cusp = Jfactor(theta20, rhogNFW, rhos_cusp, rs_cusp, 1.26)

rs_iso = 4.0
rhos_iso = rhos(rhoiso, rs_iso)
J_iso = Jfactor(theta20, rhoiso, rhos_iso, rs_iso)

print('J20 factors [GeV^2/cm^5]:')
print(f'  isothermal (rs=4): {J_iso:.3e}')
print(f'  NFW (gamma=1):     {J_nfw:.3e}')
print(f'  gNFW (gamma=1.26): {J_cusp:.3e}')

mDM = 50.0
sigmav_th = 3e-26
print(f'\nAssume mDM={mDM:.1f} GeV and thermal <sv>={sigmav_th:.1e} cm^3/s')
for ch in ['e','mu','b']:
    Ng = Ngamma(mDM, ch)
    sreq_iso = sigmav_required(phi_obs, mDM, Ng, J_iso)
    sreq_nfw = sigmav_required(phi_obs, mDM, Ng, J_nfw)
    sreq_cusp = sigmav_required(phi_obs, mDM, Ng, J_cusp)
    print(f'channel={ch:>2}, N_gamma={Ng:.4f}')
    print(f'  <sv>_req iso : {sreq_iso:.3e} cm^3/s')
    print(f'  <sv>_req nfw : {sreq_nfw:.3e} cm^3/s')
    print(f'  <sv>_req cusp: {sreq_cusp:.3e} cm^3/s')

# Optional: infer gamma (gNFW) needed for thermal b-channel at m=50 GeV
Ng_b = Ngamma(50.0, 'b')
J_needed = phi_obs * 8*np.pi*(50.0**2)/(sigmav_th*Ng_b)
def J_of_gamma(g):
    rs_g = rs(rhogNFW, g)
    rhos_g = rhos(rhogNFW, rs_g, g)
    return Jfactor(theta20, rhogNFW, rhos_g, rs_g, g)
gamma_best = bisect(lambda g: J_of_gamma(g)-J_needed, 0.8, 1.6, xtol=1e-3)
print(f'\nFor b-channel, mDM=50 GeV, thermal <sv>:')
print(f'  J_needed = {J_needed:.3e} GeV^2/cm^5')
print(f'  best-fit inner slope gamma ~ {gamma_best:.3f}')


/tmp/ipykernel_109899/3112484152.py:43: IntegrationWarning: The algorithm does not converge.  Roundoff error is detected
  in the extrapolation table.  It is assumed that the requested tolerance
  cannot be achieved, and that the returned result (if full_output = 1) is 
  the best which can be obtained.
  return kpcincm * quad(integrand, 0, 400, epsrel=1e-4)[0]


J20 factors [GeV^2/cm^5]:
  isothermal (rs=4): 2.336e+22
  NFW (gamma=1):     9.925e+22
  gNFW (gamma=1.26): 2.304e+23

Assume mDM=50.0 GeV and thermal <sv>=3.0e-26 cm^3/s
channel= e, N_gamma=0.2339
  <sv>_req iso : 1.840e-23 cm^3/s
  <sv>_req nfw : 4.331e-24 cm^3/s
  <sv>_req cusp: 1.866e-24 cm^3/s
channel=mu, N_gamma=0.1270
  <sv>_req iso : 3.389e-23 cm^3/s
  <sv>_req nfw : 7.978e-24 cm^3/s
  <sv>_req cusp: 3.437e-24 cm^3/s
channel= b, N_gamma=12.7901
  <sv>_req iso : 3.364e-25 cm^3/s
  <sv>_req nfw : 7.919e-26 cm^3/s
  <sv>_req cusp: 3.412e-26 cm^3/s


/tmp/ipykernel_109899/3112484152.py:43: IntegrationWarning: The algorithm does not converge.  Roundoff error is detected
  in the extrapolation table.  It is assumed that the requested tolerance
  cannot be achieved, and that the returned result (if full_output = 1) is 
  the best which can be obtained.
  return kpcincm * quad(integrand, 0, 400, epsrel=1e-4)[0]



For b-channel, mDM=50 GeV, thermal <sv>:
  J_needed = 2.620e+23 GeV^2/cm^5
  best-fit inner slope gamma ~ 1.287
